# Fase 1B — Preprocesamiento para Transformers (IroSvA)

Este notebook prepara las entradas utilizadas por BETO y DistilBETO.

Se generan:

- `MESSAGE_TRANSFORMER_BASE`;
- `MESSAGE_TRANSFORMER_SOCIAL`.

Se mantiene el control de calidad interno del train y el diagnóstico de tokenización.

El test se procesa con las mismas reglas, pero **no se utiliza para eliminar registros del train**.

La prevención de leakage entre folds se realiza posteriormente en cada Transformer mediante su propio `GROUP_ID` + `StratifiedGroupKFold`.


## 1. Librerías y configuración

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import emoji

try:
    from transformers import AutoTokenizer
    TRANSFORMERS_DISPONIBLE = True
except ImportError:
    AutoTokenizer = None
    TRANSFORMERS_DISPONIBLE = False

try:
    from pysentimiento.preprocessing import preprocess_tweet
    PYSENTIMIENTO_DISPONIBLE = True
except ImportError:
    preprocess_tweet = None
    PYSENTIMIENTO_DISPONIBLE = False

DATA_DIR = Path("../data")
OUT_DIR = DATA_DIR / "transformers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "DistilBETO": "dccuchile/distilbert-base-spanish-uncased",
    "RoBERTuito": "pysentimiento/robertuito-base-uncased",
}

ROBERTUITO_ALTERNATIVO = "pysentimiento/robertuito-base-deacc"

print("transformers disponible:", TRANSFORMERS_DISPONIBLE)
print("pysentimiento disponible:", PYSENTIMIENTO_DISPONIBLE)
print()
for modelo, checkpoint in CHECKPOINTS.items():
    print(f"{modelo:12} -> {checkpoint}")
print(f"{'RoBERTuito alt.':12} -> {ROBERTUITO_ALTERNATIVO}")


## 2. Carga y auditoría del train oficial

Se cargan las tres variantes dialectales por separado.

Antes de eliminar duplicados se verifica:

- cuántos textos `MESSAGE` están repetidos;
- si un mismo texto repetido tiene **la misma etiqueta**;
- si existe un **conflicto de anotación** (mismo `MESSAGE` con etiquetas distintas).

Solo los duplicados exactos de texto se deduplican después de la auditoría, manteniendo una única copia.  
Esto replica el criterio utilizado en el preprocessing de ML para mantener ambos experimentos uniformizados.


In [ ]:
def auditar_duplicados_train(df, variante):
    grupos = (
        df.groupby("MESSAGE", dropna=False)["IS_IRONIC"]
        .agg(["size", "nunique"])
        .reset_index()
    )

    repetidos = grupos[
        grupos["size"] > 1
    ].copy()

    conflictos = repetidos[
        repetidos["nunique"] > 1
    ].copy()

    mismos_labels = repetidos[
        repetidos["nunique"] == 1
    ].copy()

    return {
        "variante": variante,
        "n_original": len(df),
        "n_textos_repetidos": len(repetidos),
        "n_duplicados_misma_etiqueta": len(mismos_labels),
        "n_conflictos_etiqueta": len(conflictos),
        "mensajes_conflicto": set(
            conflictos["MESSAGE"].tolist()
        ),
        "conflictos_resumen": conflictos,
    }


def cargar_train():
    frames = []
    auditoria = []
    conflictos_detalle = []

    for variante in ["mx", "es", "cu"]:
        df = pd.read_csv(
            DATA_DIR / f"irosva.{variante}.training.csv"
        )
        df["VARIANTE"] = variante

        info = auditar_duplicados_train(
            df,
            variante,
        )

        mensajes_conflicto = info[
            "mensajes_conflicto"
        ]

        # Guardar TODAS las filas implicadas en conflictos.
        df_conflictos = df[
            df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        if not df_conflictos.empty:
            df_conflictos[
                "MOTIVO_EXCLUSION"
            ] = "mismo_texto_etiquetas_contradictorias"

            conflictos_detalle.append(
                df_conflictos
            )

            print(
                f"\n{variante.upper()} — conflictos de etiqueta:"
            )

            display(
                df_conflictos[
                    [
                        "ID",
                        "MESSAGE",
                        "IS_IRONIC",
                        "VARIANTE",
                    ]
                ].sort_values("MESSAGE")
            )

        # 1. Eliminar completamente textos con etiquetas contradictorias.
        df_sin_conflictos = df[
            ~df["MESSAGE"].isin(
                mensajes_conflicto
            )
        ].copy()

        # 2. En duplicados restantes, las etiquetas son iguales:
        #    conservar una única copia.
        antes_dedup = len(df_sin_conflictos)

        df_limpio = (
            df_sin_conflictos
            .drop_duplicates(
                subset="MESSAGE",
                keep="first",
            )
            .reset_index(drop=True)
        )

        filas_dedup_mismo_label = (
            antes_dedup
            - len(df_limpio)
        )

        auditoria.append({
            "variante": variante,
            "n_original": info["n_original"],
            "textos_repetidos": info["n_textos_repetidos"],
            "conflictos_texto": info["n_conflictos_etiqueta"],
            "filas_retiradas_conflicto": len(df_conflictos),
            "filas_duplicadas_misma_etiqueta_retiradas": filas_dedup_mismo_label,
            "n_tras_control_duplicados": len(df_limpio),
        })

        print(
            f"{variante}: "
            f"{info['n_original']} originales | "
            f"{len(df_conflictos)} filas retiradas por conflicto | "
            f"{filas_dedup_mismo_label} duplicados de misma etiqueta retirados | "
            f"{len(df_limpio)} restantes"
        )

        frames.append(
            df_limpio
        )

    df_train = pd.concat(
        frames,
        ignore_index=True,
    )

    df_auditoria = pd.DataFrame(
        auditoria
    )

    if conflictos_detalle:
        df_conflictos_train = pd.concat(
            conflictos_detalle,
            ignore_index=True,
        )
    else:
        df_conflictos_train = pd.DataFrame()

    return (
        df_train,
        df_auditoria,
        df_conflictos_train,
    )


(
    df_train,
    df_auditoria_train,
    df_conflictos_train,
) = cargar_train()

print("\nAUDITORÍA TRAIN — DUPLICADOS Y CONFLICTOS")
display(df_auditoria_train)

print(
    "\nTrain tras control de duplicados/conflictos:",
    len(df_train),
)

print(
    df_train["VARIANTE"]
    .value_counts()
    .sort_index()
)

print("\nDistribución de clases:")
print(
    df_train
    .groupby("VARIANTE")["IS_IRONIC"]
    .value_counts()
)


## 3. Carga del test oficial

El test se carga para generar las mismas representaciones BASE/SOCIAL que el train.

No se utiliza para modificar la composición del train.


In [ ]:
def cargar_test(variante):
    df_texto = pd.read_csv(
        DATA_DIR / f"irosva.{variante} - irosva.{variante}.test.csv"
    )
    df_truth = pd.read_csv(
        DATA_DIR / f"irosva.{variante}.test.truth.csv"
    )

    if "IS_IRONIC" in df_texto.columns:
        df_texto = df_texto.drop(columns=["IS_IRONIC"])

    df = df_texto.merge(
        df_truth[["ID", "IS_IRONIC"]],
        on="ID",
        how="inner",
    )

    df["VARIANTE"] = variante
    return df


df_test = pd.concat(
    [cargar_test(v) for v in ["mx", "es", "cu"]],
    ignore_index=True,
)

print("Test total:", len(df_test))
print(df_test["VARIANTE"].value_counts().sort_index())


## 5. Features lingüísticas opcionales

Se conservan las mismas 9 features del experimento ML, calculadas **desde el texto original**.

No se incorporarán en el primer baseline Transformer. Se guardan para un experimento posterior:

**Transformer puro vs Transformer + features lingüísticas**.


In [ ]:
NEGACIONES = {
    "no", "nunca", "jamas", "tampoco",
    "nada", "nadie",
    "ningun", "ninguno", "ninguna", "ningunos", "ningunas",
    "ni", "sino", "sin",
}

FEATURE_COLS = [
    "n_exc", "n_int", "n_may", "n_emo", "n_ris",
    "n_neg", "n_elo", "n_com", "n_pun",
]


def quitar_tildes(texto):
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


def extraer_features_linguisticos(texto):
    texto_str = str(texto)
    texto_norm = quitar_tildes(texto_str.lower())
    palabras = re.findall(r"\b\w+\b", texto_norm)

    n_exc = texto_str.count("!") + texto_str.count("¡")
    n_int = texto_str.count("?") + texto_str.count("¿")
    n_may = len(
        re.findall(r"\b[A-ZÁÉÍÓÚÜÑ]{3,}\b", texto_str)
    )
    n_emo = sum(
        1 for c in texto_str
        if c in emoji.EMOJI_DATA
    )
    n_ris = len(
        re.findall(
            r"\b(ja+ja+|je+je+|ji+ji+|ha+ha+|xd+|lol+|jsjs)\b",
            texto_norm,
        )
    )
    n_neg = sum(
        1 for p in palabras
        if p in NEGACIONES
    )
    n_elo = len(
        re.findall(r"(.)\1{2,}", texto_norm)
    )
    n_com = len(
        re.findall(r'["\'‘’“”«»]', texto_str)
    )
    n_pun = texto_str.count("...") + texto_str.count("…")

    return [
        n_exc, n_int, n_may, n_emo, n_ris,
        n_neg, n_elo, n_com, n_pun,
    ]


for nombre, df in [("Train", df_train), ("Test", df_test)]:
    features = df["MESSAGE"].fillna("").apply(
        extraer_features_linguisticos
    ).tolist()

    for i, col in enumerate(FEATURE_COLS):
        df[col] = [f[i] for f in features]

    print(f"{nombre}: features lingüísticas calculadas.")


## 6. Preprocessing BASE para BETO y DistilBETO

Esta versión realiza una limpieza mínima:

- Unicode y espacios normalizados.
- URL → `url`.
- mención → `@usuario`.
- elimina únicamente marcadores aislados `RT` / `FAV`.

Se conservan:

- mayúsculas;
- tildes;
- puntuación;
- `¿?` / `¡!`;
- hashtags;
- emojis;
- elongaciones;
- stopwords.

**DistilBETO es uncased:** no hace falta convertir manualmente el texto a minúsculas; se deja esa responsabilidad a su tokenizer oficial.


In [ ]:
URL_RE = re.compile(
    r"(?:https?://\S+|www\.\S+)",
    flags=re.IGNORECASE,
)

MENTION_RE = re.compile(r"(?<!\w)@\w+")

RT_FAV_RE = re.compile(
    r"\b(?:RT|FAV)\b",
    flags=re.IGNORECASE,
)


def normalizar_unicode_y_espacios(texto):
    texto = unicodedata.normalize(
        "NFKC",
        str(texto),
    )

    texto = "".join(
        c for c in texto
        if unicodedata.category(c)[0] != "C"
        or c in "\n\t"
    )

    return re.sub(r"\s+", " ", texto).strip()


def preprocesar_transformer_base(texto):
    texto = normalizar_unicode_y_espacios(texto)

    texto = URL_RE.sub(
        " url ",
        texto,
    )

    texto = MENTION_RE.sub(
        " @usuario ",
        texto,
    )

    texto = RT_FAV_RE.sub(
        " ",
        texto,
    )

    return re.sub(r"\s+", " ", texto).strip()


## 7. Preprocessing SOCIAL para BETO y DistilBETO

La literatura sobre Twitter en español deja abiertas algunas decisiones para BETO/DistilBETO.

Por eso esta segunda variante permite comprobar posteriormente, mediante validación, si ayuda:

- limitar elongaciones a 3 caracteres;
- transformar hashtags en contenido textual;
- convertir emojis a descripción textual en español.

La puntuación y las stopwords se siguen conservando.


In [ ]:
def limitar_elongaciones(
    texto,
    max_repeticiones=3,
):
    patron = re.compile(
        r"(.)\1{" + str(max_repeticiones) + r",}",
        flags=re.IGNORECASE,
    )

    return patron.sub(
        lambda m: m.group(1) * max_repeticiones,
        texto,
    )


def separar_camel_case(token):
    token = token.replace("_", " ")

    token = re.sub(
        r"(?<=[a-záéíóúüñ])(?=[A-ZÁÉÍÓÚÜÑ])",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])(?=\d)",
        " ",
        token,
    )

    token = re.sub(
        r"(?<=\d)(?=[A-Za-zÁÉÍÓÚÜÑáéíóúüñ])",
        " ",
        token,
    )

    return re.sub(
        r"\s+",
        " ",
        token,
    ).strip()


def normalizar_hashtags(texto):
    def reemplazar(match):
        contenido = separar_camel_case(
            match.group(1)
        )

        return f" hashtag {contenido} "

    return re.sub(
        r"#([\wÁÉÍÓÚÜÑáéíóúüñ]+)",
        reemplazar,
        texto,
    )


def emojis_a_texto_es(texto):
    # Ejemplo aproximado:
    # 🙄 -> :cara_con_ojos_en_blanco:
    texto = emoji.demojize(
        texto,
        language="es",
    )

    patron = re.compile(
        r":([^:\s]+):"
    )

    def reemplazar(match):
        descripcion = (
            match.group(1)
            .replace("_", " ")
        )

        return (
            f" emoji {descripcion} emoji "
        )

    return patron.sub(
        reemplazar,
        texto,
    )


def preprocesar_transformer_social(texto):
    texto = preprocesar_transformer_base(
        texto
    )

    texto = limitar_elongaciones(
        texto,
        max_repeticiones=3,
    )

    texto = normalizar_hashtags(
        texto
    )

    texto = emojis_a_texto_es(
        texto
    )

    return re.sub(
        r"\s+",
        " ",
        texto,
    ).strip()


## 8. RoBERTuito — preprocessing oficial

RoBERTuito fue entrenado específicamente para redes sociales en español.

Se utiliza directamente:

```python
pysentimiento.preprocessing.preprocess_tweet
```

en lugar de replicar manualmente su lógica.


In [ ]:
def preprocesar_robertuito(texto):
    if not PYSENTIMIENTO_DISPONIBLE:
        raise ImportError(
            "Falta pysentimiento. Instala con "
            "`pip install pysentimiento` o "
            "`uv add pysentimiento`."
        )

    return preprocess_tweet(
        str(texto),
        lang="es",
        shorten=3,
    )


## 9. Generación de textos Transformer

In [ ]:
for nombre, df in [
    ("Train", df_train),
    ("Test", df_test),
]:
    print(f"Procesando {nombre}...")

    df["MESSAGE_TRANSFORMER_BASE"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_base)
    )

    df["MESSAGE_TRANSFORMER_SOCIAL"] = (
        df["MESSAGE"]
        .fillna("")
        .astype(str)
        .apply(preprocesar_transformer_social)
    )

    if PYSENTIMIENTO_DISPONIBLE:
        df["MESSAGE_ROBERTUITO"] = (
            df["MESSAGE"]
            .fillna("")
            .astype(str)
            .apply(preprocesar_robertuito)
        )
    else:
        df["MESSAGE_ROBERTUITO"] = pd.NA
        print(
            "  AVISO: RoBERTuito no se procesó "
            "porque falta pysentimiento."
        )

print("\nEjemplos por variante:")

for variante in ["mx", "es", "cu"]:
    row = (
        df_train[
            df_train["VARIANTE"] == variante
        ]
        .iloc[0]
    )

    print(f"\n--- {variante.upper()} ---")
    print("ORIGINAL:")
    print(row["MESSAGE"])

    print("\nBASE:")
    print(row["MESSAGE_TRANSFORMER_BASE"])

    print("\nSOCIAL:")
    print(row["MESSAGE_TRANSFORMER_SOCIAL"])

    if PYSENTIMIENTO_DISPONIBLE:
        print("\nROBERTUITO:")
        print(row["MESSAGE_ROBERTUITO"])


## 10. Diagnóstico de tokenización — SOLO TRAIN

Para cada tokenizer, variante dialectal y preprocessing se calculan estadísticas descriptivas de longitud.

Este diagnóstico utiliza únicamente el train y no modifica registros.


In [ ]:
def cargar_tokenizers():
    if not TRANSFORMERS_DISPONIBLE:
        raise ImportError(
            "Falta transformers. Instala con "
            "`pip install transformers` o "
            "`uv add transformers`."
        )

    tokenizers = {
        "BETO": AutoTokenizer.from_pretrained(
            CHECKPOINTS["BETO"],
            use_fast=True,
        ),
        "DistilBETO": AutoTokenizer.from_pretrained(
            CHECKPOINTS["DistilBETO"],
            use_fast=True,
        ),
    }

    if PYSENTIMIENTO_DISPONIBLE:
        tokenizers["RoBERTuito"] = AutoTokenizer.from_pretrained(
            CHECKPOINTS["RoBERTuito"],
            use_fast=True,
        )

    return tokenizers

def calcular_stats_tokenizacion(
    tokenizer,
    textos,
):
    longitudes = []
    unk = 0
    total_tokens = 0
    total_palabras = 0

    unk_id = tokenizer.unk_token_id

    for texto in textos:
        texto = str(texto)

        encoded = tokenizer(
            texto,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )

        ids = encoded["input_ids"]

        longitudes.append(
            len(ids)
        )

        total_tokens += len(ids)
        total_palabras += max(
            len(texto.split()),
            1,
        )

        if unk_id is not None:
            unk += sum(
                token == unk_id
                for token in ids
            )

    arr = np.asarray(
        longitudes,
        dtype=np.int32,
    )

    return {
        "n": len(arr),
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "p99": np.percentile(arr, 99),
        "max": int(arr.max()),
        "pct_gt_64": 100 * (arr > 64).mean(),
        "pct_gt_128": 100 * (arr > 128).mean(),
        "tokens_por_palabra": (
            total_tokens
            / max(total_palabras, 1)
        ),
        "unk_por_1000_tokens": (
            1000 * unk
            / max(total_tokens, 1)
        ),
    }


def diagnosticar_train():
    tokenizers = cargar_tokenizers()

    configs = [
        (
            "BETO_BASE",
            "BETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "BETO_SOCIAL",
            "BETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
        (
            "DISTILBETO_BASE",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_BASE",
        ),
        (
            "DISTILBETO_SOCIAL",
            "DistilBETO",
            "MESSAGE_TRANSFORMER_SOCIAL",
        ),
    ]

    if PYSENTIMIENTO_DISPONIBLE:
        configs.append(
            (
                "ROBERTUITO",
                "RoBERTuito",
                "MESSAGE_ROBERTUITO",
            )
        )

    filas = []

    for cfg, tokenizer_name, columna in configs:
        tok = tokenizers[tokenizer_name]

        for variante in ["mx", "es", "cu"]:
            textos = (
                df_train.loc[
                    df_train["VARIANTE"] == variante,
                    columna,
                ]
                .fillna("")
                .astype(str)
                .tolist()
            )

            stats = calcular_stats_tokenizacion(
                tok,
                textos,
            )

            filas.append({
                "configuracion": cfg,
                "tokenizer": tokenizer_name,
                "variante": variante,
                **stats,
            })

    return pd.DataFrame(filas)


if TRANSFORMERS_DISPONIBLE:
    df_tokenizacion = diagnosticar_train()

    display(
        df_tokenizacion.style.format({
            "p50": "{:.1f}",
            "p95": "{:.1f}",
            "p99": "{:.1f}",
            "pct_gt_64": "{:.2f}%",
            "pct_gt_128": "{:.2f}%",
            "tokens_por_palabra": "{:.3f}",
            "unk_por_1000_tokens": "{:.3f}",
        })
    )
else:
    df_tokenizacion = None
    print(
        "Instala transformers para ejecutar "
        "el diagnóstico."
    )


## 11. Exportación final por variante

Los CSV exportados son la entrada común de BETO y DistilBETO.

El preprocessing no elimina registros del train en función del contenido del test.


In [ ]:
COLUMNAS_TEXTO = [
    "MESSAGE_TRANSFORMER_BASE",
    "MESSAGE_TRANSFORMER_SOCIAL",
    "MESSAGE_ROBERTUITO",
]

COLUMNAS_SALIDA = (
    [
        "ID",
        "TOPIC",
        "IS_IRONIC",
        "MESSAGE",
        "VARIANTE",
    ]
    + FEATURE_COLS
    + COLUMNAS_TEXTO
)

print("=" * 78)
print("EXPORTACIÓN FINAL — TRANSFORMERS")
print("=" * 78)

for split, df in [
    ("train", df_train),
    ("test", df_test),
]:
    for variante in ["mx", "es", "cu"]:
        salida = (
            df.loc[
                df["VARIANTE"] == variante,
                COLUMNAS_SALIDA,
            ]
            .reset_index(drop=True)
        )

        ruta = (
            OUT_DIR
            / f"{split}_transformers_{variante}.csv"
        )

        salida.to_csv(
            ruta,
            index=False,
        )

        print(
            f"[EXPORT][{split.upper()}][{variante.upper()}] "
            f"{len(salida)} registros -> {ruta}"
        )


if df_tokenizacion is not None:
    ruta_diag = (
        OUT_DIR
        / "diagnostico_tokenizacion_train.csv"
    )

    df_tokenizacion.to_csv(
        ruta_diag,
        index=False,
    )

    print(
        "\n[EXPORT][AUDIT] Diagnóstico tokenización ->",
        ruta_diag,
    )


# Auditoría interna del train.
df_auditoria_train.to_csv(
    OUT_DIR / "auditoria_duplicados_train.csv",
    index=False,
)

if len(df_conflictos_train) > 0:
    df_conflictos_train.to_csv(
        OUT_DIR / "conflictos_etiqueta_train.csv",
        index=False,
    )


# Manifest mínimo: solo permite comprobar versión y cantidad de filas.
manifest_rows = []

for variante in ["mx", "es", "cu"]:
    manifest_rows.append({
        "pipeline_version": "transformers_comun_v1",
        "variante": variante,
        "train_final": int(
            (df_train["VARIANTE"] == variante).sum()
        ),
        "test_n": int(
            (df_test["VARIANTE"] == variante).sum()
        ),
    })

manifest = pd.DataFrame(manifest_rows)

manifest_path = (
    OUT_DIR
    / "manifest_preprocesamiento_transformers.csv"
)

manifest.to_csv(
    manifest_path,
    index=False,
)

print("\nMANIFEST DEL PREPROCESSING")
display(manifest)

print("\n[PREPROCESS][Transformers] EXPORTACIÓN: OK")


## 12. Uso posterior

BETO y DistilBETO cargan directamente:

```text
data/transformers/train_transformers_mx.csv
data/transformers/train_transformers_es.csv
data/transformers/train_transformers_cu.csv
```

Cada modelo construye después su propio `GROUP_ID` utilizando sus representaciones y su tokenizer.

El preprocessing no decide los folds ni elimina registros en función del test.


## 13. Resumen metodológico

### Train

- control interno de duplicados/conflictos;
- generación BASE/SOCIAL;
- diagnóstico de tokenización;
- exportación.

### Modelos

BETO y DistilBETO:

1. cargan el train preprocesado;
2. construyen `GROUP_ID` propio;
3. ejecutan `StratifiedGroupKFold` 5×3;
4. seleccionan hiperparámetros con F1-Macro.

### Test

Se mantiene como conjunto de evaluación final y no determina qué registros permanecen en train.
